In [ ]:
import pandas as pd # data loading and manipulation
import numpy as np # numerical operations
import matplotlib.pyplot as plt  # base plotting
import seaborn as sns  # styled plotting
import os # used to build file paths
import joblib # saving and loading models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
# Load the four processed CSV files
# X = features (30 columns: V1-V28, Log_Amount, Hour)
# y = target (0 = legitimate, 1 = fraud)
X_train = pd.read_csv(os.path.join('..', 'data', 'processed', 'X_train.csv'))
y_train = pd.read_csv(os.path.join('..', 'data', 'processed', 'y_train.csv')).squeeze()

X_test = pd.read_csv(os.path.join('..', 'data', 'processed', 'X_test.csv'))
y_test = pd.read_csv(os.path.join('..', 'data', 'processed', 'y_test.csv')).squeeze()

lr = joblib.load(os.path.join('..', 'models', 'logistic_regression.pkl'))
rf = joblib.load(os.path.join('..', 'models', 'random_forest.pkl'))

In [ ]:
predict_lr_test = lr.predict(X_test)
predict_rf_test = rf.predict(X_test)

In [ ]:
confusion_matrix_lr = confusion_matrix(y_test, predict_lr_test)
display_cm = ConfusionMatrixDisplay(confusion_matrix_lr, display_labels=['Legitimate', 'Fraud'])
display_cm.plot(colorbar=False, color='white')
plt.title('Logistic Regression (Confusion Matrix)')

# Add TP/TN/FP/FN labels to each cell
plt.text(0, 0.2, 'TN', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(1, 0.2, 'FP', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(0, 1.2, 'FN', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(1, 1.2, 'TP', ha='center', va='center', color='white', fontsize=14, fontweight='bold')

plt.show()

In [ ]:
confusion_matrix_rf = confusion_matrix(y_test, predict_rf_test)
display_cm = ConfusionMatrixDisplay(confusion_matrix_rf, display_labels=['Legitimate', 'Fraud'])
display_cm.plot(colorbar=False)
plt.title('Random Forest (Confusion Matrix)')

# Add TP/TN/FP/FN labels to each cell
plt.text(0, 0.2, 'TN', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(1, 0.2, 'FP', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(0, 1.2, 'FN', ha='center', va='center', color='white', fontsize=14, fontweight='bold')
plt.text(1, 1.2, 'TP', ha='center', va='center', color='white', fontsize=14, fontweight='bold')

plt.show()

In [ ]:
lr_probs = lr.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]

lr_precision, lr_recall, _ = precision_recall_curve(y_test, lr_probs)
rf_precision, rf_recall, _ = precision_recall_curve(y_test, rf_probs)

plt.figure(figsize=(8, 6))
plt.grid(True, alpha=0.3)
plt.plot(lr_recall, lr_precision, label=f'Logistic Regression (AUC = {auc(lr_recall, lr_precision):.2f})')
plt.plot(rf_recall, rf_precision, label=f'Random Forest (AUC = {auc(rf_recall, rf_precision):.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

In [ ]:

lr_false_positive_rate, lr_true_positive_rate, _ = roc_curve(y_test, lr_probs)
rf_false_positive_rate, rf_true_positive_rate, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8, 6))
plt.grid(True, alpha=0.3)
plt.plot(lr_false_positive_rate, lr_true_positive_rate, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, lr_probs):.2f})')
plt.plot(rf_false_positive_rate, rf_true_positive_rate, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:

# We apply our own threshold manually

threshold = 0.3
predict_rf_custom = (rf_probs >= threshold).astype(int)

print('Random Forest — Default Threshold (0.5)')
print('='*50)
print(classification_report(y_test, predict_rf_test, target_names=['Legitimate', 'Fraud']))

print(f'Random Forest — Custom Threshold (threshold = {threshold})')
print('='*50)
print(classification_report(y_test, predict_rf_custom, target_names=['Legitimate', 'Fraud']))

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.1)
precisions = []
recalls = []

for t in thresholds:
    predict_rf_custom = (rf_probs >= t).astype(int)
    precisions.append(precision_score(y_test, predict_rf_custom))
    recalls.append(recall_score(y_test, predict_rf_custom))

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions, 'o-', color='steelblue', linewidth=2, markersize=6, label='Precision')
plt.plot(thresholds, recalls, 's-', color='crimson', linewidth=2, markersize=6, label='Recall')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision and Recall vs Threshold (Random Forest)',  fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Random Forest"],
    "Threshold": [0.5, 0.5, threshold],
    "Precision": [ precision_score(y_test, predict_lr_test), precision_score(y_test, predict_rf_test), precision_score(y_test, predict_rf_custom)],
    "Recall": [recall_score(y_test, predict_lr_test), recall_score(y_test, predict_rf_test), recall_score(y_test, predict_rf_custom)],
    "F1-Score": [f1_score(y_test, predict_lr_test), f1_score(y_test, predict_rf_test), f1_score(y_test, predict_rf_custom)]
})

results = results.round(2)
print(results.to_string(index=False))
